# Day 17: Vectorizing, two heaps, backprop by hand

*Week 4: ML from scratch*

**Time budget:** about 60–90 minutes. **A** is Python fluency (~15 min), **B** is an interview-style algorithm (~25 min), **C** is NumPy / PyTorch / ML (~30 min).

**For each problem:**
1. Plan on paper or in comments before you type any code.
2. Work on it for 20 minutes before asking for a hint.
3. Run the test cell under it. Don't edit the tests.
4. Answer the **Say out loud** prompts, out loud. Interviews grade the explanation as much as the code.

In [ ]:
import numpy as np

## A. Vectorize this

`NumPy`

This function standardizes each **column** of `X` (subtract the column's mean, divide by its standard deviation) using Python loops:

```python
def slow_standardize(X):
    n, d = X.shape
    out = np.zeros_like(X, dtype=float)
    for j in range(d):
        col = [X[i, j] for i in range(n)]
        mean = sum(col) / n
        var = sum((v - mean) ** 2 for v in col) / n
        std = var ** 0.5
        for i in range(n):
            out[i, j] = (X[i, j] - mean) / (std + 1e-8)
    return out
```

Write `fast_standardize(X)` with the same output and **no Python loops**. The test also checks that it's at least 10× faster.

**Look up:** `axis=0`, broadcasting a (d,) vector across (n, d).

In [ ]:
def fast_standardize(X):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
import time
def slow_standardize(X):
    n, d = X.shape
    out = np.zeros_like(X, dtype=float)
    for j in range(d):
        col = [X[i, j] for i in range(n)]
        mean = sum(col) / n
        var = sum((v - mean) ** 2 for v in col) / n
        std = var ** 0.5
        for i in range(n):
            out[i, j] = (X[i, j] - mean) / (std + 1e-8)
    return out
X = np.random.default_rng(0).normal(3, 2, size=(2000, 50))
t0 = time.perf_counter(); slow = slow_standardize(X); t_slow = time.perf_counter() - t0
t0 = time.perf_counter(); fast = fast_standardize(X); t_fast = time.perf_counter() - t0
assert np.allclose(slow, fast)
assert t_slow / max(t_fast, 1e-9) > 10, f"only {t_slow / t_fast:.1f}x faster"
print(f"✅ A passed ({t_slow / t_fast:.0f}x faster)")

## B. Running median with two heaps

`Algorithms`

Write a class `MedianFinder` with `add(num)` and `median()`. `add` should be O(log n) and `median` O(1).

Keep two heaps: a **max-heap** for the smaller half and a **min-heap** for the larger half. After every `add`, rebalance so that their sizes differ by at most one. The median is then at the top of one or both heaps.

**Look up:** Python's `heapq` only has a min-heap. Store negated numbers to get a max-heap.

In [ ]:
class MedianFinder:
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
import random
mf = MedianFinder()
mf.add(1); assert mf.median() == 1
mf.add(2); assert mf.median() == 1.5
mf.add(3); assert mf.median() == 2
random.seed(0)
mf, seen = MedianFinder(), []
for _ in range(300):
    x = random.randint(-100, 100)
    mf.add(x); seen.append(x)
    assert mf.median() == np.median(seen), (len(seen), mf.median(), np.median(seen))
print("✅ B passed")

## C. Backprop by hand for a 2-layer network

`ML from scratch`

**The** classic ML research interview question. The network is:

```
h   = relu(X @ W1 + b1)      X: (N, D)   W1: (D, H)   b1: (H,)
out = h @ W2 + b2            W2: (H, O)  b2: (O,)
loss = mean((out - y) ** 2)  mean over all N*O entries
```

1. `forward(params, X, y)`: return `(loss, cache)`. `cache` holds whatever the backward pass needs.
2. `backward(params, cache)`: return a dict of gradients with the same keys as `params` ("W1", "b1", "W2", "b2") and the same shapes.

Work through it on paper first, from the loss backwards: d loss/d out → dW2, db2, dh → through the ReLU → dW1, db1. **Check every shape.** A gradient always has the same shape as the thing it's the gradient of, and that tells you where the transposes go.

The test compares your gradients with PyTorch autograd.

In [ ]:
def forward(params, X, y):
    # your code here
    pass


def backward(params, cache):
    # your code here
    pass

In [ ]:
# Tests: run after your solution. Don't edit.
import torch
rng = np.random.default_rng(0)
N, D, H, O = 8, 5, 6, 3
params = {"W1": rng.normal(size=(D, H)), "b1": rng.normal(size=H),
          "W2": rng.normal(size=(H, O)), "b2": rng.normal(size=O)}
X, y = rng.normal(size=(N, D)), rng.normal(size=(N, O))
loss, cache = forward(params, X, y)
grads = backward(params, cache)
tp = {k: torch.tensor(v, requires_grad=True) for k, v in params.items()}
h = torch.relu(torch.tensor(X) @ tp["W1"] + tp["b1"])
tloss = ((h @ tp["W2"] + tp["b2"] - torch.tensor(y)) ** 2).mean()
tloss.backward()
assert np.isclose(loss, tloss.item()), "forward loss is wrong"
for k in params:
    assert grads[k].shape == params[k].shape, f"{k}: shape {grads[k].shape} vs {params[k].shape}"
    assert np.allclose(grads[k], tp[k].grad.numpy()), f"gradient for {k} is wrong"
print("✅ C passed")

## Wrap-up (5 minutes)

- For each problem: what's its time and space complexity? Which edge cases did you handle?
- Anything that took more than 20 minutes, or needed a hint: add it to the **Redo list** in `README.md` and solve it again from scratch in 3 days, without looking at today's code.
- Tick off today in `README.md`.